# Exercise 3 — run_paper_trader

The bot loop iterates bar by bar. On a 0→1 signal transition it buys; on a 1→0 transition it sells. At each bar it records the current portfolio value. At the end, any open position is force-liquidated so the account is fully settled.

In [ ]:
import pandas as pd, math
from dataclasses import dataclass, field

def _synthetic(n=252):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })
@dataclass
class Trade:
    date:        object
    action:      str
    price:       float
    shares:      float
    cash_after:  float
    value_after: float
@dataclass
class PaperAccount:
    initial_cash: float = 10_000.0
    cash:         float = field(init=False)
    shares:       float = field(init=False)
    trades:       list  = field(init=False)

    def __post_init__(self):
        self.cash   = self.initial_cash
        self.shares = 0.0
        self.trades = []

    def portfolio_value(self, price):
        return self.cash + self.shares * float(price)

    def buy(self, date, price, fraction=1.0):
        price = float(price)
        if self.cash <= 0 or price <= 0:
            return None
        shares = (self.cash * fraction) / price
        cost   = shares * price
        if cost > self.cash:
            shares = self.cash / price
            cost   = shares * price
        self.cash   -= cost
        self.shares += shares
        t = Trade(date=date, action="BUY", price=price, shares=shares,
                  cash_after=self.cash,
                  value_after=self.portfolio_value(price))
        self.trades.append(t)
        return t

    def sell(self, date, price):
        price = float(price)
        if self.shares <= 0:
            return None
        proceeds    = self.shares * price
        sold_shares = self.shares
        self.cash  += proceeds
        self.shares = 0.0
        t = Trade(date=date, action="SELL", price=price, shares=sold_shares,
                  cash_after=self.cash,
                  value_after=self.portfolio_value(price))
        self.trades.append(t)
        return t

def run_paper_trader(df, signals, initial_cash=10_000.0, fraction=1.0):
    """Simulate paper trading bar by bar.

    Algorithm:
      account     = PaperAccount(initial_cash)
      prev_signal = 0
      for i in range(len(df)):
          sig = int(signals.iloc[i])
          if sig == 1 and prev_signal == 0:  → buy
          elif sig == 0 and prev_signal == 1: → sell
          append portfolio_value to eq_values
          prev_signal = sig
      if account.shares > 0: → force-sell at last close
      compute equity Series, total_return, max_drawdown
      return result dict

    Returns dict with:
        account, trades, equity, initial_cash, final_value,
        total_return, max_drawdown, n_trades, n_buys, n_sells
    """
    account     = PaperAccount(initial_cash=initial_cash)
    eq_values   = []
    prev_signal = 0

    for i in range(len(df)):
        date  = df.index[i]
        price = float(df["Close"].iloc[i])
        sig   = int(signals.iloc[i])
        # TODO: buy on 0→1, sell on 1→0
        eq_values.append(account.portfolio_value(price))
        prev_signal = sig

    # TODO: force-sell if open position
    equity       = pd.Series(eq_values, index=df.index)
    total_return = float(equity.iloc[-1] / initial_cash - 1.0)
    peak         = equity.cummax()
    max_dd       = float(((equity - peak) / peak).min())
    return {
        "account":      account,
        "trades":       account.trades,
        "equity":       equity,
        "initial_cash": initial_cash,
        "final_value":  float(equity.iloc[-1]),
        "total_return": total_return,
        "max_drawdown": max_dd,
        "n_trades":     len(account.trades),
        "n_buys":       sum(1 for t in account.trades if t.action == "BUY"),
        "n_sells":      sum(1 for t in account.trades if t.action == "SELL"),
    }


### Checks

In [ ]:
checks = 0

# 1 — flat signal: equity = initial_cash throughout
try:
    df  = _synthetic()
    sig = pd.Series(0, index=df.index)
    r   = run_paper_trader(df, sig, initial_cash=10_000.0)
    assert (r["equity"] == 10_000.0).all(), "flat signal → equity always equals initial_cash"
    assert r["n_trades"]  == 0
    assert r["n_buys"]    == 0
    assert r["n_sells"]   == 0
    checks += 1; print("✅ 1 flat signal → equity=10000 throughout, 0 trades")
except Exception as e:
    print("❌ 1:", e)

# 2 — always-long signal: 1 BUY + 1 forced SELL = 2 trades
try:
    df  = _synthetic()
    sig = pd.Series(1, index=df.index)
    r   = run_paper_trader(df, sig, initial_cash=10_000.0)
    assert r["n_buys"]  == 1, f"expected 1 buy, got {r['n_buys']}"
    assert r["n_sells"] == 1, f"expected 1 sell, got {r['n_sells']}"
    checks += 1; print("✅ 2 always-long: 1 BUY + 1 forced SELL = 2 total trades")
except Exception as e:
    print("❌ 2:", e)

# 3 — always-long equity tracks Close / initial_price × initial_cash
try:
    df  = _synthetic()
    sig = pd.Series(1, index=df.index)
    r   = run_paper_trader(df, sig, initial_cash=10_000.0)
    # Bought at close[0]: shares = 10000 / close[0]
    shares0  = 10_000.0 / df["Close"].iloc[0]
    expected = df["Close"] * shares0
    diff     = (r["equity"] - expected).abs().max()
    assert diff < 1e-6, f"equity tracks price × shares, max diff = {diff}"
    checks += 1; print("✅ 3 equity tracks Close × shares bought at entry")
except Exception as e:
    print("❌ 3:", e)

# 4 — equity Series has correct length and index
try:
    df  = _synthetic()
    sig = pd.Series(1, index=df.index)
    r   = run_paper_trader(df, sig)
    assert len(r["equity"]) == len(df)
    assert (r["equity"].index == df.index).all()
    checks += 1; print("✅ 4 equity Series has same length and index as df")
except Exception as e:
    print("❌ 4:", e)

# 5 — total_return = final_value / initial_cash - 1
try:
    df  = _synthetic()
    sig = pd.Series(1, index=df.index)
    r   = run_paper_trader(df, sig, initial_cash=10_000.0)
    expected_tr = r["final_value"] / 10_000.0 - 1.0
    assert abs(r["total_return"] - expected_tr) < 1e-9
    assert r["max_drawdown"] <= 1e-9   # always ≤ 0
    checks += 1; print("✅ 5 total_return = final_value/initial_cash - 1; max_drawdown ≤ 0")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
